# Example notebook: plot aerial imagery from Rotterdam Hub

In [ ]:
from pprint import pprint
import requests
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt

List all datasets

In [ ]:
response = requests.get('https://hub.clearly.app/api/datasets?query={"ownerHubId":"65e9c5c8fe2ac522c6ac7b43"}')
response.raise_for_status()
content = response.content.decode()
pprint(content)

Retrieve information about WMS dataset ID

In [ ]:
response = requests.get("https://hub.clearly.app/api/datasets/65ef1e39fe2ac522c6ac7b5e")
response.raise_for_status()
content = response.content.decode()
pprint(content)

Use URL to access data

In [ ]:
WMS_URL = "https://diensten.rotterdam.nl/arcgis/services/LUCHTFOTO/luchtfoto_actueel/MapServer/WMSServer"

# Define a bounding box around Rotterdam (EPSG:4326, lon/lat order for WMS 1.1.1)
bbox = "4.45,51.90,4.55,51.95"

params = {
    "service": "WMS",
    "request": "GetMap",
    "version": "1.1.1",
    "layers": "0",
    "styles": "",
    "srs": "EPSG:4326",
    "bbox": bbox,
    "width": 800,
    "height": 600,
    "format": "image/png",
    "transparent": "true",
}

# Request the image
response = requests.get(WMS_URL, params=params, timeout=30)
response.raise_for_status()

content_type = response.headers.get("Content-Type", "")
print("Status:", response.status_code)
print("Content-Type:", content_type)

if "image" not in content_type.lower():
    preview = response.text[:500]
    raise ValueError(f"WMS did not return an image. Response preview:\n{preview}")

Plot sample image

In [ ]:
# Load image into PIL
image = Image.open(BytesIO(response.content))

# Display
plt.imshow(image)
plt.axis("off")
plt.title("Rotterdam Aerial Photography (WMS)")
plt.show()